# DV1583 - Tree Motion User Study Analysis

## Imports, loading and cleaning

In [30]:
# Module imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

# Plot style
sns.set(style="whitegrid")

In [76]:
# Load CSV
filePath = "TreeMotionUserDataTest.csv"

df = pd.read_csv(filePath, encoding="cp1252") # Windows enconding for latin. Needed to specify for reading. Swedish moment.
df.columns = df.columns.str.strip()
df.head()

# Data cleaning
# Remove unnecessary columns
drop_cols = ["Starttid", "Slutförandetid", "E-post", "Namn", 
             "I hereby agree to participate in this study voluntarily. I have read the explanation of the study, I understand that my response will be anonymous, and I may withdraw at any time without providing a r",
            "DEMO - Rate how natural the motion of the tree appears to be..The motion of the tree felt..."]
df = df.drop(columns=drop_cols)

# Rename columns with questions to simple descriptions
df = df.rename(columns=
               {"What is your gender?": "gender", "What is your age group?": "age_group", 
                "How familiar are you with video games or other computer-generated visual content (e.g 3D animations, VFX, or other digital graphics)?": "familiarity"})

df = df.rename(columns={"Were you in a relatively distraction-free environment during the study?": "distraction_free",
                       "Did the videos play smoothly without lag or stuttering?": "video_smooth",
                       "Was the visual quality of the videos sufficient for you to judge the motions?": "video_quality"})

# Rename clip question columns
new_columns = {}

for col in df.columns:
    # Search and rename rating columns
    match_rating = re.search(r"Clip\s*(\d+).*felt", col.strip(), re.IGNORECASE)
    if match_rating:
        clip_num = match_rating.group(1)
        new_columns[col] = f"clip_{clip_num}_rating"

    # Search and rename optional text responses
    match_text = re.search(r"Clip\s*(\d+).*What made", col.strip(), re.IGNORECASE)
    if match_text:
        clip_num = match_text.group(1)
        new_columns[col] = f"clip_{clip_num}_comment"
        
df = df.rename(columns=new_columns)

# Convert comment columns into strings
comment_cols = []

for col in df.columns:
    if "_comment" in col:
        comment_cols.append(col)

df[comment_cols] = df[comment_cols].astype("string")

# Convert rating columns
rating_cols = []
for col in df.columns:
    if "_rating" in col:
        rating_cols.append(col)

id_cols = ["Id", "gender", "age_group", "familiarity", "distraction_free", "video_smooth", "video_quality"]

# Melt columns (Convert from wide to long)
df_long = df.melt(id_vars=id_cols, value_vars=rating_cols, var_name="clip", value_name="rating")

# Extract clip number
df_long["clip_num"] = df_long["clip"].str.extract(r"(\d+)").astype("int")

# Convert Likert to numeric values
likert_map = {
    "Very unnatural": 1,
    "Unnatural": 2,
    "Neutral": 3,
    "Natural": 4,
    "Very natural": 5}

df_long["rating"] = df_long["rating"].str.strip()
df_long["rating_num"] = df_long["rating"].map(likert_map)
#df_long[["Id", "clip_num"]].head(10)

#df.info()
#df.columns
#df.head()

df_long.info()
df_long.columns
df_long.head()

<class 'pandas.DataFrame'>
RangeIndex: 54 entries, 0 to 53
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Id                54 non-null     int64
 1   gender            54 non-null     str  
 2   age_group         54 non-null     str  
 3   familiarity       54 non-null     str  
 4   distraction_free  54 non-null     str  
 5   video_smooth      54 non-null     str  
 6   video_quality     54 non-null     str  
 7   clip              54 non-null     str  
 8   rating            54 non-null     str  
 9   clip_num          54 non-null     int64
 10  rating_num        54 non-null     int64
dtypes: int64(3), str(8)
memory usage: 4.8 KB


,Id,gender,age_group,familiarity,distraction_free,video_smooth,video_quality,clip,rating,clip_num,rating_num
0,1,Man,25-34,Extremely familiar,Yes,Mostly,"Yes, completely",clip_1_rating,Natural,1,4
1,2,Woman,25-34,Very familiar,Yes,Mostly,"Yes, completely",clip_1_rating,Natural,1,4
2,1,Man,25-34,Extremely familiar,Yes,Mostly,"Yes, completely",clip_2_rating,Unnatural,2,2
3,2,Woman,25-34,Very familiar,Yes,Mostly,"Yes, completely",clip_2_rating,Very unnatural,2,1
4,1,Man,25-34,Extremely familiar,Yes,Mostly,"Yes, completely",clip_3_rating,Natural,3,4


## Descriptive Statistics (Mean and Median)